# Yield Curves and Forward Rates

A single yield per bond (01/01) is a convenient summary, but money lent for one year and money lent for ten years earn different rates.
The yield curve gives a rate for every maturity. From it follow discount factors, the rates implied today for borrowing in the future
(forward rates), and the par yields at which new bonds would be issued. The link between them is an integral: the zero rate for
maturity $T$ is the average of the instantaneous forward rates between now and $T$. This notebook builds a smooth Swiss-franc-like curve,
moves between the three representations, bootstraps zero rates from bond prices, and prices bonds off the curve.

## 0. Math link

- [Introduction to integrals](https://github.com/dnlwrthstr/LearningCalculus/blob/main/hypersklill/04_integrals/00_introduction_to_integrals.ipynb):
  the zero rate times maturity is the area under the forward-rate curve, $z(T)\,T = \int_0^T f(s)\,ds$, and the zero rate is the
  average height of that curve.
- [Derivatives of polynomials](https://github.com/dnlwrthstr/LearningCalculus/blob/main/hypersklill/02_derivaties/01_derivatives_of_polynomials.ipynb)
  and the product rule: the forward rate is the derivative $f(T) = \frac{d}{dT}\big[z(T)\,T\big] = z(T) + T z'(T)$.
- Continuous compounding from [00/02](../00_returns_and_compounding/02_continuous_compounding_as_a_limit.ipynb) and bond prices from
  [01/01](./01_bond_price_as_a_function_of_yield.ipynb).

## 1. Introduction

The Swiss National Bank, the Confederation's debt office and every bank's treasury maintain a CHF yield curve. A bank pricing a
five-year fixed mortgage needs today's five-year rate; a pension fund discounting a payment in 23 years needs the 23-year rate; a trader
wondering whether the market expects the SNB to cut rates reads the forward curve. All three are views of the same object.

After this notebook you can:

- convert between discount factors, zero rates, forward rates and par yields;
- derive forward rates as the derivative of $z(T)T$, and zero rates as averages of forward rates;
- bootstrap zero rates from coupon bond prices;
- price a coupon bond from the curve and compare with pricing at a single yield;
- read an upward-sloping and an inverted curve, and explain what forward rates do and do not predict.

## 2. Theory

### Discount factors and zero rates

The **discount factor** $d(T)$ is today's price of CHF 1 paid at time $T$ (a zero-coupon bond with face 1). The **zero rate** $z(T)$ is the
continuously compounded yield of that zero:

$$
d(T) = e^{-z(T)\,T}, \qquad z(T) = -\frac{\ln d(T)}{T} .
$$

A bond with cash flows $C_t$ is worth $P = \sum_t C_t\, d(t)$: each cash flow is discounted at the rate for its own maturity.

### Forward rates

Lending from $T_1$ to $T_2$, agreed today, must earn the rate that makes it equivalent to lending to $T_2$ directly versus lending to $T_1$
and rolling over (otherwise there is an arbitrage):

$$
e^{z(T_2)T_2} = e^{z(T_1)T_1}\, e^{f(T_1, T_2)(T_2 - T_1)} \quad\Rightarrow\quad f(T_1, T_2) = \frac{z(T_2)T_2 - z(T_1)T_1}{T_2 - T_1} .
$$

Letting $T_2 \to T_1 = T$ gives the **instantaneous forward rate**, a derivative:

$$
f(T) = \frac{d}{dT}\big[z(T)\,T\big] = z(T) + T\, z'(T) = -\frac{d}{dT}\ln d(T) .
$$

### Zero rates as averages of forwards

Integrating (with $z(0)\cdot 0 = 0$) gives the inverse relation:

$$
z(T)\,T = \int_0^T f(s)\,ds, \qquad z(T) = \frac{1}{T}\int_0^T f(s)\,ds, \qquad d(T) = \exp\Big(-\int_0^T f(s)\,ds\Big) .
$$

The zero rate is the average forward rate up to $T$. Consequences:

- if the curve slopes up ($z' > 0$), then $f > z$: the forward curve lies above the zero curve;
- if it slopes down (inverted), $f < z$;
- the forward curve crosses the zero curve where the zero curve is flat (a maximum or minimum).

### Par yields

The **par yield** $c(T)$ is the annual coupon at which a bond maturing at $T$ is priced at par:

$$
c(T) \sum_{t=1}^{T} d(t) + d(T) = 1 \quad\Rightarrow\quad c(T) = \frac{1 - d(T)}{\sum_{t=1}^{T} d(t)} .
$$

It is a weighted average of the one-year forward rates (annually compounded), with weights proportional to the discount factors, so on an
upward-sloping curve it lies a little below the zero curve.

### Bootstrapping

Given prices of coupon bonds maturing in years $1, 2, \dots$, solve for the discount factors one at a time: the one-year bond gives
$d(1)$; the two-year bond, with $d(1)$ known, gives $d(2)$; and so on. Each step solves one linear equation.

### A smooth curve: Nelson–Siegel

Central banks fit curves of the Nelson–Siegel form

$$
z(T) = \beta_0 + \beta_1 \frac{1 - e^{-T/\tau}}{T/\tau} + \beta_2\Big(\frac{1 - e^{-T/\tau}}{T/\tau} - e^{-T/\tau}\Big),
$$

whose three terms are level, slope and curvature (the same three factors PCA finds in yield changes, 07/04). Its forward curve is
$f(T) = \beta_0 + \beta_1 e^{-T/\tau} + \beta_2 \frac{T}{\tau}e^{-T/\tau}$.

## 3. Worked Examples

### Example 1: from zero rates to forwards

Zero rates (continuous): 1 year 0.50%, 2 years 0.70%. The forward rate from year 1 to year 2 is
$(0.70\% \times 2 - 0.50\% \times 1)/1 = 0.90\%$. Lending for two years at 0.70% is the same as lending for one year at 0.50% and then
one more year at 0.90%.

### Example 2: discount factors and a bond

With $z(1) = 0.50\%$, $z(2) = 0.70\%$, $z(3) = 0.85\%$: $d(1) = 0.99501$, $d(2) = 0.98610$, $d(3) = 0.97482$. A three-year 1% bond is
worth $1 \times 0.99501 + 1 \times 0.98610 + 101 \times 0.97482 = 100.437$. Its single yield to maturity (annual compounding) is about
0.85%: a blend of the three rates, dominated by the last one.

### Example 3: bootstrapping two bonds

A one-year 0.5% bond trades at 99.90, a two-year 1% bond at 100.50. Then $d(1) = 99.90/100.5 = 0.99403$ and
$1 \times 0.99403 + 101\, d(2) = 100.50$ gives $d(2) = 0.98521$, so $z(2) = -\ln(0.98521)/2 = 0.745\%$.

### Example 4: an inverted curve

If the one-year rate is 3% and the two-year rate 2.5% (continuous), the forward from year 1 to 2 is $2 \times 2.5\% - 3\% = 2\%$. The market
prices in lower short rates next year, typical before a central bank cuts.

## 4. Finance Application

### Where it matters

- **Pricing.** Every fixed-income instrument (bonds, swaps, mortgages, the fixed leg of a pension promise) is valued with discount factors
  from a curve, not with a single yield.
- **Forward-rate agreements and futures.** SARON futures and FRAs lock in forward rates; their fair prices come straight from the curve.
- **Monetary policy.** Forward rates show where the market expects policy rates to go, plus a term premium.
- **Risk.** Key-rate durations measure sensitivity to moves of individual points on the curve, beyond parallel shifts (01/02).
- **Liabilities.** Pension funds and insurers discount liabilities with the risk-free curve (Swiss Solvency Test, IFRS 17).

### The usual misreadings

- "The forward rate is the market's forecast." It is a forecast plus a term premium; historically, forwards have overpredicted future short
  rates on average.
- "A bond's yield is the rate for its maturity." It is an average over all its cash-flow dates, weighted by present value.
- "An inverted curve always means recession." It often precedes one, but the curve also reflects term premia and central-bank asset
  purchases.
- "Zero rates, par yields and forward rates are interchangeable." They coincide only for a flat curve.

## 5. Python Examples

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

### A synthetic CHF curve (Nelson–Siegel)

Parameters chosen to resemble a moderately upward-sloping Swiss curve: short rates near 0.4%, long rates near 1.3%. Rates are continuously
compounded, in decimals.

In [ ]:
beta0, beta1, beta2, tau = 0.013, -0.009, 0.004, 3.0

def ns_loading(T):
    x = np.asarray(T, dtype=float) / tau
    return np.where(x > 0, (1 - np.exp(-x)) / np.where(x > 0, x, 1), 1.0)

def zero_rate(T):
    L = ns_loading(T)
    return beta0 + beta1 * L + beta2 * (L - np.exp(-np.asarray(T) / tau))

def forward_rate(T):
    x = np.asarray(T, dtype=float) / tau
    return beta0 + beta1 * np.exp(-x) + beta2 * x * np.exp(-x)

def discount(T):
    return np.exp(-zero_rate(T) * np.asarray(T))

def par_yield(T):
    t = np.arange(1, T + 1)
    return (1 - discount(T)) / discount(t).sum()

maturities = np.array([1, 2, 3, 5, 7, 10, 15, 20, 30])
pd.DataFrame({"zero rate": zero_rate(maturities), "instantaneous forward": forward_rate(maturities),
              "discount factor": discount(maturities), "par yield (annual)": [par_yield(T) for T in maturities]},
             index=pd.Index(maturities, name="maturity (years)"))

On this upward-sloping curve the forward rates lie above the zero rates. The par yields (annual compounding) look almost identical to
the continuous zero rates in the table because two small effects offset: annual compounding raises the number, the coupon effect lowers
it. Put on the same convention, par yields lie slightly below zero rates, by about 0.7 bp at 10 years and 1 bp at 30 years.

### The derivative and the integral, checked numerically

In [ ]:
T_grid = np.linspace(0, 30, 30_001)
h = T_grid[1] - T_grid[0]

# forward rate as the derivative of z(T) T
zT = zero_rate(T_grid) * T_grid
f_numeric = np.gradient(zT, h)
print("max |f_numeric - f_formula| :", np.abs(f_numeric[1:-1] - forward_rate(T_grid[1:-1])).max())

# zero rate as the average of forward rates (cumulative trapezoid rule)
f_vals = forward_rate(T_grid)
area = np.concatenate([[0.0], np.cumsum((f_vals[1:] + f_vals[:-1]) / 2 * h)])
z_from_f = area[1:] / T_grid[1:]
print("max |z_from_f - z_formula|  :", np.abs(z_from_f - zero_rate(T_grid[1:])).max())

Both directions agree to about $10^{-9}$ or better: differentiating $z(T)T$ gives the forward curve, and averaging the forward curve gives
back the zero curve.

### Forward rates between dates

In [ ]:
def forward_between(T1, T2):
    return (zero_rate(T2) * T2 - zero_rate(T1) * T1) / (T2 - T1)

pairs = [(1, 2), (2, 3), (1, 5), (5, 10), (10, 20), (20, 30)]
pd.DataFrame({"forward rate": [forward_between(a, b) for a, b in pairs]},
             index=pd.Index([f"{a}y -> {b}y" for a, b in pairs], name="period"))

### Bootstrapping zero rates from bond prices

We generate prices of annual-coupon bonds maturing in 1 to 10 years from the curve, add a little noise (bid–ask, rounding to 0.01), and
bootstrap the discount factors back.

In [ ]:
rng = np.random.default_rng(42)
coupons = np.array([0.25, 0.5, 0.75, 1.0, 1.0, 1.25, 1.5, 1.5, 1.75, 2.0]) / 100   # annual coupon rates
n_years = len(coupons)
true_prices = np.array([coupons[k] * 100 * discount(np.arange(1, k + 2)).sum() + 100 * discount(k + 1)
                        for k in range(n_years)])
quoted = np.round(true_prices + rng.normal(0, 0.01, n_years), 2)

d_boot = np.empty(n_years)
for k in range(n_years):
    coupon_cf = coupons[k] * 100
    known = coupon_cf * d_boot[:k].sum()
    d_boot[k] = (quoted[k] - known) / (coupon_cf + 100)

years = np.arange(1, n_years + 1)
boot = pd.DataFrame({"quoted price": quoted, "bootstrapped d": d_boot, "true d": discount(years),
                     "bootstrapped zero": -np.log(d_boot) / years, "true zero": zero_rate(years)},
                    index=pd.Index(years, name="maturity"))
boot["error (bp)"] = (boot["bootstrapped zero"] - boot["true zero"]) * 1e4
boot

Bootstrapping recovers the zero curve to within about a basis point. The errors are largest at the short end: a price error of 0.01 on a
one-year bond is 1 bp of yield, on a ten-year bond about 0.1 bp, because the longer bond's price is more sensitive to its yield.

### Pricing off the curve against pricing at one yield

In [ ]:
def price_from_curve(coupon, T):
    t = np.arange(1, T + 1)
    cf = np.full(T, coupon * 100.0)
    cf[-1] += 100
    return np.sum(cf * discount(t))

def ytm(price, coupon, T):
    y = 0.01
    t = np.arange(1, T + 1)
    cf = np.full(T, coupon * 100.0)
    cf[-1] += 100
    for _ in range(50):
        f = np.sum(cf / (1 + y) ** t) - price
        fp = -np.sum(t * cf / (1 + y) ** (t + 1))
        y -= f / fp
    return y

rows = []
for coupon in [0.0, 0.01, 0.04]:
    p = price_from_curve(coupon, 10)
    rows.append({"coupon": coupon, "price from curve": p, "yield to maturity (annual)": ytm(p, coupon, 10),
                 "10y zero rate (annual)": np.expm1(zero_rate(10))})
pd.DataFrame(rows).set_index("coupon")

Three ten-year bonds priced off the same curve have different yields to maturity: the higher the coupon, the more weight on early cash
flows discounted at lower short rates, and the lower the yield. This "coupon effect" is why comparing bonds by yield alone can mislead
when the curve is not flat.

## 6. Visualizations

In [ ]:
T_plot = np.linspace(0.05, 30, 400)
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(T_plot, zero_rate(T_plot) * 100, label="zero rate z(T)")
ax.plot(T_plot, forward_rate(T_plot) * 100, label="instantaneous forward f(T)")
par_T = np.arange(1, 31)
ax.plot(par_T, [np.log1p(par_yield(T)) * 100 for T in par_T], "o", ms=3, label="par yield (converted to continuous)")
ax.plot(boot.index, boot["bootstrapped zero"] * 100, "kx", label="bootstrapped zero rates")
ax.set_title("A synthetic CHF yield curve (continuously compounded)")
ax.set_xlabel("maturity (years)")
ax.set_ylabel("rate (% per year)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

The forward curve rises fastest and lies above the zero curve wherever the zero curve slopes up. Par yields lie just below the zero
curve. The bootstrapped points sit on the zero curve.

In [ ]:
T_star = 10
s = np.linspace(0, 30, 400)
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(s, forward_rate(s) * 100, label="forward rate f(s)")
mask = s <= T_star
ax.fill_between(s[mask], 0, forward_rate(s[mask]) * 100, alpha=0.3,
                label=f"area = z({T_star}) x {T_star} = {zero_rate(T_star) * T_star * 100:.2f} %-years")
ax.axhline(zero_rate(T_star) * 100, xmax=T_star / 30, color="C1", ls="--",
           label=f"average height z({T_star}) = {zero_rate(T_star):.3%}")
ax.set_title("The zero rate is the average of the forward rates up to T")
ax.set_xlabel("time s (years)")
ax.set_ylabel("rate (% per year)")
ax.set_xlim(0, 30)
ax.set_ylim(0, None)
ax.legend(loc="lower right")
ax.grid(True, alpha=0.3)
plt.show()

The shaded area under the forward curve up to ten years equals $z(10) \times 10$; the dashed line at the ten-year zero rate is the
rectangle with the same area. The discount factor is $d(10) = e^{-\text{area}}$.

In [ ]:
beta_inv = (0.012, 0.018, -0.006, 1.5)        # high short rates, falling: an inverted curve

def zero_rate_ns(T, b0, b1, b2, t_):
    x = np.asarray(T, dtype=float) / t_
    L = (1 - np.exp(-x)) / x
    return b0 + b1 * L + b2 * (L - np.exp(-x))

def forward_rate_ns(T, b0, b1, b2, t_):
    x = np.asarray(T, dtype=float) / t_
    return b0 + b1 * np.exp(-x) + b2 * x * np.exp(-x)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(T_plot, zero_rate_ns(T_plot, *beta_inv) * 100, label="zero rate")
ax.plot(T_plot, forward_rate_ns(T_plot, *beta_inv) * 100, label="instantaneous forward")
ax.set_title("An inverted curve: forwards below zero rates")
ax.set_xlabel("maturity (years)")
ax.set_ylabel("rate (% per year)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

When short rates are high and expected to fall, the zero curve slopes down and the forward curve lies below it. Here the zero curve
falls over the whole 30 years, so the two never cross; the forward curve bottoms out near 1.2% after about five years, the long-run
rate the market prices in once the expected cuts are done.

## 7. Exercises

### Basic

1. Continuous zero rates are 1.0% (1 year), 1.4% (2 years) and 1.6% (3 years). Compute the discount factors and the forward rates for years
   1–2 and 2–3.
2. Using the synthetic curve, price a five-year bond with a 2% annual coupon and find its yield to maturity.
3. The one-year-in-one-year forward rate on the synthetic curve is above today's one-year rate. A commentator says: "The market expects the SNB
   to hike." **Interpret** this claim. What else could explain the difference?

### Intermediate

4. Show numerically that the par yield is the discount-factor-weighted average of the annual one-year forward rates $d(t-1)/d(t) - 1$.
5. Bootstrap the curve again with price noise of standard deviation 0.05 instead of 0.01. How large do the errors in the zero rates become,
   and where? What happens to the implied one-year forward rates?

### Advanced

6. Prove that $f(T) > z(T)$ whenever $z'(T) > 0$, and that $f$ and $z$ cross exactly where $z$ has a stationary point.
7. Compute key-rate sensitivities of a ten-year 1.5% bond: bump the zero curve by 1 bp only in a triangular window around 2, 5 and 10 years
   (and zero elsewhere), and report the price change for each. Check that the three sum to the parallel DV01 when the windows add up to 1.

## 8. Solutions / Checks

In [ ]:
# 1.
z1 = np.array([0.010, 0.014, 0.016])
t1 = np.array([1, 2, 3])
d1 = np.exp(-z1 * t1)
print("1. discount factors", np.round(d1, 5), " forwards", np.round(np.diff(z1 * t1), 4))

# 2.
p2 = price_from_curve(0.02, 5)
print(f"2. price {p2:.4f}, yield to maturity {ytm(p2, 0.02, 5):.4%}")

# 3.
print(f"3. one-year rate {zero_rate(1):.3%}, 1y-in-1y forward {forward_between(1, 2):.3%}")

# 4.
for T in [5, 10, 30]:
    t = np.arange(1, T + 1)
    d_t, d_prev = discount(t), discount(t - 1)
    fwd_annual = d_prev / d_t - 1
    print(f"4. T = {T}: par yield {par_yield(T):.6f}, weighted forwards {np.sum(d_t * fwd_annual) / d_t.sum():.6f}")

**3.** The forward is the rate that can be locked in today; it equals the expected future rate only if investors demand no premium for
lending long. A positive term premium (compensation for the risk of holding long bonds) makes forwards exceed expected rates even if no hike
is expected. The spread is evidence of expectations only after an estimate of the term premium.

**4.** $\sum_t d(t)\,(d(t-1)/d(t) - 1) = \sum_t (d(t-1) - d(t)) = 1 - d(T)$, which is the par-yield numerator: the identity is exact, as the
numbers confirm.

In [ ]:
# 5.
quoted5 = np.round(true_prices + rng.normal(0, 0.05, n_years), 2)
d5 = np.empty(n_years)
for k in range(n_years):
    cpn = coupons[k] * 100
    d5[k] = (quoted5[k] - cpn * d5[:k].sum()) / (cpn + 100)
z5 = -np.log(d5) / years
fwd5 = np.diff(np.concatenate([[0], z5 * years]))
print("5. zero errors (bp):", np.round((z5 - zero_rate(years)) * 1e4, 1))
print("   one-year forward errors (bp):", np.round((fwd5 - np.diff(np.concatenate([[0], zero_rate(years) * years]))) * 1e4, 1))

# 7.
def bond_price_with_bump(bump_fn, coupon=0.015, T=10):
    t = np.arange(1, T + 1)
    cf = np.full(T, coupon * 100.0)
    cf[-1] += 100
    return np.sum(cf * np.exp(-(zero_rate(t) + bump_fn(t)) * t))

nodes = [2, 5, 10]
def tent(k):
    # piecewise-linear "hat" functions on the nodes, flat beyond the ends; they sum to 1
    def fn(t):
        return np.interp(t, nodes, np.eye(len(nodes))[k]) * 1e-4
    return fn

base = bond_price_with_bump(lambda t: 0 * t)
krd = [bond_price_with_bump(tent(k)) - base for k in range(3)]
parallel = bond_price_with_bump(lambda t: 1e-4 + 0 * t) - base
print("7. key-rate price changes (CHF per 100):", np.round(krd, 5), " sum", round(sum(krd), 5), " parallel", round(parallel, 5))

**5.** Zero-rate errors grow roughly fivefold and are largest at short maturities. Forward rates are differences of $z(T)T$, so neighbouring
errors subtract and the one-year forwards are much noisier than the zero rates: forward curves from raw bootstrapping are jagged, which is
why smooth fitted curves such as Nelson–Siegel are used.

**6.** $f - z = T z'(T)$, which has the sign of $z'$ for $T > 0$ and is zero exactly where $z' = 0$.

**7.** The ten-year node carries most of the sensitivity (the final payment), the two- and five-year nodes the coupons. Because the hat
functions add up to 1 at every maturity, the three key-rate changes sum to the parallel change (up to second-order terms).

## 9. Conclusion

- Discount factors, zero rates, forward rates and par yields are four views of one curve; $d(T) = e^{-z(T)T} = e^{-\int_0^T f}$.
- The forward rate is the derivative of $z(T)T$, and the zero rate is the average forward rate: a derivative–integral pair.
- On an upward-sloping curve forwards lie above zero rates; on an inverted curve below.
- Bootstrapping solves for discount factors one maturity at a time; noise is amplified in forward rates.
- Bonds should be priced off the curve; a single yield to maturity mixes rates and depends on the coupon.

This completes module 01. Next: [02/01 Mean, median, mode of fund returns and AuM](../02_describing_returns/01_mean_median_mode_of_fund_returns_and_aum.ipynb).